# fak — Tier 2: the fused in-kernel model (neocloud notebook)

This is **Notebook B** — the GPU-deep companion to `fak-quickstart.ipynb`. It runs the
deepest fusion: the model runs **inside the kernel's address space**, decoding over a
kernel-owned KV cache, reachable via `/v1/fak/syscall`. It then fronts a **bigger** model
and exposes a **stable endpoint** an external client can reach — the two things a free,
no-ingress Colab can't do.

Host it on a neocloud Jupyter with a real terminal and port ingress — **Lightning AI Studio**
or a **RunPod / Vast** GPU pod (the `[VERIFIED]` single-GPU rows in
`PLAN-cloud-neocloud-rightsizing-2026-06-20.md`).

| Section | What it shows | Needs |
|---|---|---|
| **2a — synthetic in-kernel** | the real prefill+decode loop over a kernel-owned KV cache | **CPU ok** |
| **2b — real SmolLM2-135M** | the same path on weights exported from HuggingFace | **CPU ok** (Python/torch) |
| **bigger model + endpoint** | `qwen2.5:14b` behind `fak serve` on a 24 GB card, exposed publicly | **GPU** (24 GB) |

> Honest scope (matches `CLAIMS.md`): the in-kernel paths are proven correct at the
> tensor layer against a HuggingFace oracle — they make model state first-class kernel-owned
> state; they are **not** a production chat-quality serving engine. For practical chat, the
> bigger-model section uses Tier 1 (front Ollama).

> _Generated by `tools/gen_notebooks.py` — edit the generator, not this `.ipynb`. CI guards drift + stale references via `--check`._

In [ ]:
# --- setup: work dir, repo coords, GPU ---
import os, shutil, subprocess, sys, time, json, urllib.request

WORK = os.environ.get("FAK_WORK") or ("/content" if os.path.isdir("/content")
        else "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/fak-demo")
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)

REPO   = os.environ.get("FAK_REPO", "anthony-chaudhary/fak")
BRANCH = os.environ.get("FAK_BRANCH", "main")  # pin a release tag here, e.g. v0.30.0

HAS_GPU = shutil.which("nvidia-smi") is not None and \
          subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("work dir :", WORK)
print(f"repo     : {REPO}@{BRANCH}")
print("GPU      :", "yes" if HAS_GPU else "no — Tier 2 in-kernel still runs on CPU; the bigger-model section is skipped")

In [ ]:
# --- get a fresh `fak` binary plus the examples/testdata the demos use.
#     The Go module is the repository ROOT (go.mod), so the clone dir *is* the build dir.
#     By default an existing clone is refreshed to FAK_BRANCH. Set FAK_REFRESH=0 to reuse it.
import urllib.request, tarfile, platform, re

def sh(cmd, timeout=900):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, timeout=timeout)

def go_ok():
    try:
        out = subprocess.run(["go","version"], capture_output=True, text=True, check=True).stdout
        m = re.search(r"go(\d+)\.(\d+)", out)
        return bool(m) and (int(m.group(1)), int(m.group(2))) >= (1, 26)
    except Exception:
        return False

def latest_go_archive():
    arch = platform.machine().lower()
    goarch = {"x86_64":"amd64", "amd64":"amd64", "aarch64":"arm64", "arm64":"arm64"}.get(arch)
    if sys.platform.startswith("linux") and goarch:
        rels = json.load(urllib.request.urlopen("https://go.dev/dl/?mode=json", timeout=30))
        for rel in rels:
            m = re.match(r"go(\d+)\.(\d+)", rel.get("version", ""))
            if not m or (int(m.group(1)), int(m.group(2))) < (1, 26):
                continue
            for f in rel.get("files", []):
                if f.get("kind") == "archive" and f.get("os") == "linux" and f.get("arch") == goarch:
                    return rel["version"], f["filename"]
    return "go1.26.3", "go1.26.3.linux-amd64.tar.gz"

if not go_ok():
    GOV, GO = latest_go_archive()
    print("installing", GOV, "...")
    urllib.request.urlretrieve("https://go.dev/dl/" + GO, "/tmp/" + GO)
    shutil.rmtree("/usr/local/go", ignore_errors=True)
    with tarfile.open("/tmp/" + GO) as t: t.extractall("/usr/local")
os.environ["PATH"] = "/usr/local/go/bin:" + os.environ["PATH"]
print(subprocess.run(["go","version"], capture_output=True, text=True).stdout.strip())

# clone the public repo (anonymous; no token needed), then refresh to the requested ref
FAK_DIR = os.path.join(WORK, "fak")
if not os.path.isdir(os.path.join(FAK_DIR, ".git")):
    print(f"$ git clone --depth 1 -b {BRANCH} https://github.com/{REPO}.git  ->  {FAK_DIR}")
    subprocess.run(["git","clone","--depth","1","-b",BRANCH,
                    f"https://github.com/{REPO}.git", FAK_DIR], check=True)
else:
    print("clone exists:", FAK_DIR)
    if os.environ.get("FAK_REFRESH", "1") != "0":
        sh(f'git -C "{FAK_DIR}" fetch --depth 1 origin "{BRANCH}"')
        sh(f'git -C "{FAK_DIR}" checkout -q --detach FETCH_HEAD')
    else:
        print("FAK_REFRESH=0, reusing the existing checkout without fetch")

stamp = subprocess.run(["git","-C",FAK_DIR,"log","-1","--format=%h %cI %s"],
                       capture_output=True, text=True, check=True).stdout.strip()
print("source  :", stamp)

# the module is the repo root, so build right here
FAK_EXE = "fak.exe" if os.name == "nt" else "fak"
FAK = os.path.join(FAK_DIR, FAK_EXE)
sh(f'cd "{FAK_DIR}" && go build -o "{FAK_EXE}" ./cmd/fak')
print("\nfak ->", FAK)
print(subprocess.run([FAK,"version"], capture_output=True, text=True).stdout.strip() or "built ok")

In [ ]:
# --- helpers: (re)start a fak serve on a port, and call the in-kernel syscall ---
def serve(args, port=8137, env=None):
    subprocess.run('pkill -f "fak serve" 2>/dev/null; sleep 1', shell=True)
    e = dict(os.environ)
    if env: e.update(env)
    p = subprocess.Popen([FAK,"serve","--addr",f"127.0.0.1:{port}",*args],
                         cwd=FAK_DIR, env=e, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(90):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/healthz", timeout=2) as r:
                print("fak serve up:", r.read().decode()); return p
        except Exception: time.sleep(1)
    print("did not bind on", port); return p

def syscall(tool, arguments, port=8137):
    # NB: the fak-native wire key is "arguments" (NOT "args" — unknown keys are dropped)
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/fak/syscall",
        data=json.dumps({"tool":tool,"arguments":arguments}).encode(),
        headers={"Content-Type":"application/json"})
    return json.load(urllib.request.urlopen(req, timeout=120))

print("helpers ready")

## 2a — synthetic in-kernel checkpoint (instant, CPU ok)

`--engine inkernel` with no weights runs a small deterministic synthetic checkpoint. The
tokens are meaningless (random-init weights) — what it proves is the **real prefill+decode
loop over the kernel-owned KV cache**, dispatched through the adjudicator.

In [ ]:
serve(["--engine","inkernel","--model","smollm2-inkernel"], port=8137)
out = syscall("read_file", {"path":"notes.txt"}, port=8137)
print("verdict :", out.get("verdict"))
print("meta    :", out.get("result", {}).get("meta"))
print("content :", out.get("result", {}).get("content", "")[:200])

## 2b — real SmolLM2-135M weights (CPU torch is enough)

`scripts/fetch-model.sh` makes a Python venv, installs torch/transformers (CPU is fine),
downloads `HuggingFaceTB/SmolLM2-135M-Instruct`, and exports it into
`internal/model/.cache/smollm2-135m`. First run needs network to HuggingFace; it takes a
few minutes. `FAK_MODEL_DIR` is what selects the real weights.

In [ ]:
try:
    print("exporting real weights (one-time, ~mins) ...")
    subprocess.run(f'cd "{FAK_DIR}" && ./scripts/fetch-model.sh', shell=True, check=True)
    MODEL_DIR = os.path.join(FAK_DIR, "internal/model/.cache/smollm2-135m")
    serve(["--engine","inkernel","--model","smollm2-135m"], port=8137,
          env={"FAK_MODEL_DIR": MODEL_DIR})
    out = syscall("read_file", {"path":"notes.txt"}, port=8137)
    print("verdict :", out.get("verdict"))
    print("meta    :", out.get("result", {}).get("meta"))
except subprocess.CalledProcessError as e:
    print("fetch-model failed (needs Python 3.10+ and network to HuggingFace):", e)
    print("the synthetic path in 2a already exercised the in-kernel decode loop.")

## Bigger model + a stable endpoint (needs a 24 GB GPU)

A free Colab T4 (16 GB) tops out around a 7B model and has **no inbound port**. On a
neocloud 24 GB card you can front `qwen2.5:14b` (~9 GB) through the kernel — the same shape
as the live GCP demo, but reader-owned — and expose it so an external client (or Claude
Code) can point at it.

In [ ]:
if not HAS_GPU:
    print("no GPU — skipping. (Tier 2 in-kernel above ran on CPU.)")
else:
    MODEL = os.environ.get("FAK_MODEL", "qwen2.5:14b")  # ~9 GB, fits a 24 GB card
    if shutil.which("ollama") is None:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["ollama","serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try: urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
        except Exception: time.sleep(1)
    print("pulling", MODEL, "(one-time, cached) ...")
    subprocess.run(["ollama","pull",MODEL], check=True)
    serve(["--base-url","http://127.0.0.1:11434/v1","--model",MODEL], port=8080)

    subprocess.run([sys.executable,"-m","pip","install","-q","openai"], check=True)
    from openai import OpenAI
    client = OpenAI(base_url="http://127.0.0.1:8080/v1", api_key="not-needed")
    r = client.chat.completions.create(model=MODEL, temperature=0.3,
        messages=[{"role":"user","content":"In one sentence, what is a syscall?"}])
    print("\nmodel (through the kernel):", r.choices[0].message.content)

In [ ]:
# Expose :8080 publicly. On Lightning AI: use the Studio "expose port" control.
# On RunPod: the pod's proxy URL is https://<POD_ID>-8080.proxy.runpod.net .
# Portable fallback (no signup) — a cloudflared quick tunnel; watch for the trycloudflare URL:
if HAS_GPU:
    try:
        subprocess.run(
            "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
            "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
            shell=True, check=True)
        print("starting cloudflared quick tunnel to :8080 — the https://*.trycloudflare.com")
        print("URL will print below; point any OpenAI client at  <that-url>/v1")
        subprocess.Popen(["cloudflared","tunnel","--url","http://localhost:8080"])
        time.sleep(8)
    except Exception as e:
        print("tunnel optional — on Lightning/RunPod use the platform's port-expose instead:", e)

## Where to go next

- **Why this is the deep fusion:** `docs/explainers/addressable-kv-cache.md` (mid-run causal
  span eviction, bit-exact `max|Δ| = 0`).
- **The expert smoke** — Qwen3.6-27B through fak's own GGUF→Q8 path (`cmd/fakchat`):
  `GETTING-STARTED.md` §4c–4d.
- **The cost/hardware tiers** behind this notebook: `PLAN-cloud-neocloud-rightsizing-2026-06-20.md`.
- **What's real vs. simulated vs. stub:** `CLAIMS.md`.

*Cleanup:* `!pkill -f "fak serve"; pkill -f ollama; pkill -f cloudflared`